# EDA – gresspollen

- undersøke manglende verdier
- se på fordelingen av gresspollen
- undersøke hvordan gresspollen varierer over tid og mellom år
- se på sammenhenger mellom pollen og værvariabler

In [5]:
import pandas as pd
import matplotlib.pyplot as plt

 1. Leser inn datasettet:

In [6]:
df = pd.read_csv(
    "../data/full_pollen_and_weather_data_Zurich.csv",
    sep=";"
)

df.shape

(32352, 46)

2. Gir kolonnene mer forståelige navn og velger relevante variabler:

In [7]:
df = df.rename(columns={
    "station_abbr_pollen": "Pollenstasjon",
    "station_abbr_weather": "Værstasjon",
    "master_timestamp": "Tid",
    "timestamp_pollen": "Pollen_tid",
    "timestamp_weather": "Vær_tid",
    "kabetuh0": "Bjørk",
    "khpoach0": "Gress",
    "tre200h0": "Lufttemperatur",
    "fkl010h0": "Vindstyrke",
    "ure200h0": "Luftfuktighet",
    "rre150h0": "Nedbør",
    "sre000h0": "Solskinn",
    "dkl010h0": "Vindretning"
})

data = df[[
    "Tid",
    "Bjørk",
    "Lufttemperatur",
    "Vindstyrke",
    "Luftfuktighet",
    "Solskinn",
    "Nedbør"
]].copy()

3. Undersøker rådata:

In [8]:
print("Størrelse:", data.shape)
print("\nDatatyper:")
print(data.dtypes)

print("\nManglende verdier:")
print(data.isna().sum())

print("\nAntall duplikater:")
print(data.duplicated().sum())

Størrelse: (32352, 7)

Datatyper:
Tid                   str
Bjørk             float64
Lufttemperatur    float64
Vindstyrke        float64
Luftfuktighet     float64
Solskinn            int64
Nedbør            float64
dtype: object

Manglende verdier:
Tid                 0
Bjørk             779
Lufttemperatur      0
Vindstyrke          7
Luftfuktighet       0
Solskinn            0
Nedbør              0
dtype: int64

Antall duplikater:
0


4. Konverterer tidspunkt og lager daglige verdier:

In [9]:
data["Tid"] = pd.to_datetime(data["Tid"])
data["dato"] = data["Tid"].dt.date

daglig_alle = data.groupby("dato").agg(
    bjørk_mean=("Bjørk", "mean"),
    bjørk_antall=("Bjørk", "count"),

    temperatur_mean=("Lufttemperatur", "mean"),
    temperatur_antall=("Lufttemperatur", "count"),

    vind_mean=("Vindstyrke", "mean"),
    vind_antall=("Vindstyrke", "count"),

    luftfuktighet_mean=("Luftfuktighet", "mean"),
    luftfuktighet_antall=("Luftfuktighet", "count"),

    solskinn_total=("Solskinn", "sum"),
    solskinn_antall=("Solskinn", "count"),

    nedbør_total=("Nedbør", "sum"),
    nedbør_antall=("Nedbør", "count")
).reset_index()

daglig_alle["dato"] = pd.to_datetime(daglig_alle["dato"])
daglig_alle["dag_i_året"] = daglig_alle["dato"].dt.dayofyear

5. Filtrerer dager med for få målinger:

In [10]:
daglig_gyldige = daglig_alle[
    (daglig_alle["bjørk_antall"] >= 20) &
    (daglig_alle["temperatur_antall"] >= 20) &
    (daglig_alle["vind_antall"] >= 20) &
    (daglig_alle["luftfuktighet_antall"] >= 20) &
    (daglig_alle["solskinn_antall"] >= 20) &
    (daglig_alle["nedbør_antall"] >= 20)
].copy()

daglig_gyldige = daglig_gyldige.sort_values("dato").copy()

print("Før filtrering:", daglig_alle.shape)
print("Etter filtrering:", daglig_gyldige.shape)

Før filtrering: (1348, 14)
Etter filtrering: (1301, 14)


6. Sjekker manglende verdier etter cleaning:

In [11]:
daglig_gyldige.isna().sum()

dato                    0
bjørk_mean              0
bjørk_antall            0
temperatur_mean         0
temperatur_antall       0
vind_mean               0
vind_antall             0
luftfuktighet_mean      0
luftfuktighet_antall    0
solskinn_total          0
solskinn_antall         0
nedbør_total            0
nedbør_antall           0
dag_i_året              0
dtype: int64

7. Statistikk:

In [12]:
daglig_gyldige[[
    "bjørk_mean",
    "temperatur_mean",
    "vind_mean",
    "luftfuktighet_mean",
    "solskinn_total",
    "nedbør_total"
]].describe()

,bjørk_mean,temperatur_mean,vind_mean,luftfuktighet_mean,solskinn_total,nedbør_total
count,1301.000000,1301.000000,1301.000000,1301.000000,1301.000000,1301.000000
mean,18.534678,11.644126,2.022198,73.911984,308.381245,3.051422
std,101.122703,7.657262,1.194586,13.503430,273.903114,6.263040
min,0.000000,-6.854167,0.333333,32.445833,0.000000,0.000000
25%,0.000000,6.116667,1.250000,65.233333,39.000000,0.000000
50%,0.000000,11.229167,1.712500,75.295833,268.000000,0.000000
75%,0.000000,17.991667,2.375000,84.462500,542.000000,3.200000
max,1787.083333,29.291667,8.833333,99.183333,853.000000,45.900000


8. Bjørkpollen over tid:

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(
    daglig_gyldige["dato"],
    daglig_gyldige["bjørk_mean"]
)

plt.xlabel("Dato")
plt.ylabel("Gjennomsnittlig bjørkepollenkonsentrasjon")
plt.title("Daglig bjørkepollenkonsentrasjon i Zürich")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

9. Bjørkepollen i et spesifikt år:


In [ ]:
år = 2024

data_år = daglig_gyldige[
    daglig_gyldige["dato"].dt.year == år
].copy()

plt.figure(figsize=(12, 6))

plt.plot(
    data_år["dato"],
    data_år["bjørk_mean"]
)

plt.xlabel("Dato")
plt.ylabel("Gjennomsnittlig bjørkepollenkonsentrasjon")
plt.title(f"Bjørkepollen i Zürich - {år}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


10. Sammenligner bjørkepollen mellom år:


In [ ]:
daglig_gyldige["år"] = daglig_gyldige["dato"].dt.year

bjork_per_år = daglig_gyldige.groupby("år")["bjørk_mean"].agg(
    ["mean", "median", "max"]
)

bjork_per_år


11. Fordeling av bjørkepollen:


In [ ]:
plt.figure(figsize=(10, 5))

plt.hist(
    daglig_gyldige["bjørk_mean"].dropna(),
    bins=40
)

plt.xlabel("Gjennomsnittlig bjørkepollenkonsentrasjon")
plt.ylabel("Antall dager")
plt.title("Fordeling av daglige bjørkepollenverdier")
plt.tight_layout()
plt.show()


12. Korrelasjon mellom pollen og værvariabler:


In [ ]:
korrelasjon = daglig_gyldige[
    [
        "bjørk_mean",
        "temperatur_mean",
        "vind_mean",
        "luftfuktighet_mean",
        "solskinn_total",
        "nedbør_total"
    ]
].corr()

korrelasjon


13. Bjørkepollen mot temperatur:


In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    daglig_gyldige["temperatur_mean"],
    daglig_gyldige["bjørk_mean"],
    alpha=0.5
)

plt.xlabel("Gjennomsnittlig temperatur")
plt.ylabel("Gjennomsnittlig bjørkepollenkonsentrasjon")
plt.title("Bjørkepollen og temperatur")
plt.tight_layout()
plt.show()


14. Bjørkepollen mot nedbør:

In [ ]:
plt.figure(figsize=(8, 6))

plt.scatter(
    daglig_gyldige["nedbør_total"],
    daglig_gyldige["bjørk_mean"],
    alpha=0.5
)

plt.xlabel("Daglig nedbør")
plt.ylabel("Gjennomsnittlig bjørkepollenkonsentrasjon")
plt.title("Bjørkepollen og nedbør")
plt.tight_layout()
plt.show()
